# 03 - Model Training

This notebook trains and compares spam classifiers:

- Logistic Regression (the `src/train.py` baseline)
- Multinomial Naive Bayes
- Linear SVM

It evaluates each on the held-out test set, picks the winner, calibrates its
probabilities (Brier score), exports it to `models/spam_model.pkl`, saves
metrics and the confusion matrix to `results/`, and demoes a prediction.

> **Prerequisite:** datasets exist at `data/raw/sms_spam.csv` and `data/raw/email_spam.csv`.

In [ ]:
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

# Allow importing the project's src modules regardless of where Jupyter was launched
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from preprocessing import load_email_data, load_sms_data, preprocess_data  # noqa: E402

%matplotlib inline


## 1. Load and split the data

In [ ]:
sms = load_sms_data()
email = load_email_data()
df = preprocess_data(pd.concat([sms, email], ignore_index=True))
print(f"Total messages: {len(df):,}")

X_train, X_test, y_train, y_test = train_test_split(
    df["cleaned_message"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)
# Hold out 20% of the training set for probability calibration
X_train, X_cal, y_train, y_cal = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)
print(f"Train: {len(X_train):,} | Calibration: {len(X_cal):,} | Test: {len(X_test):,}")

## 2. Train and compare models

All models use the same TF-IDF features (10,000, unigrams + bigrams) so the
comparison is apples-to-apples.

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer(max_features=10_000, ngram_range=(1, 2))),
        ("clf", LogisticRegression(max_iter=1000)),
    ]),
    "Naive Bayes": Pipeline([
        ("tfidf", TfidfVectorizer(max_features=10_000, ngram_range=(1, 2))),
        ("clf", MultinomialNB()),
    ]),
    "Linear SVM": Pipeline([
        ("tfidf", TfidfVectorizer(max_features=10_000, ngram_range=(1, 2))),
        ("clf", LinearSVC(max_iter=2000)),
    ]),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    if hasattr(model, "decision_function"):
        scores = model.decision_function(X_test)
    else:
        scores = model.predict_proba(X_test)[:, 1]
    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "f1_spam": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, scores),
    })

summary = pd.DataFrame(results).set_index("model").round(3)
summary

In [ ]:
best_name = summary["f1_spam"].idxmax()
best_model = models[best_name]
print(f"Best model by spam F1: {best_name}")
print(classification_report(y_test, best_model.predict(X_test), target_names=["ham", "spam"]))

In [ ]:
ConfusionMatrixDisplay.from_estimator(
    best_model, X_test, y_test, display_labels=["ham", "spam"], cmap="Blues"
)
plt.title(f"Confusion matrix — {best_name}")
plt.show()

## 3. Calibrate probabilities

LinearSVC has no native `predict_proba`, so `src/predict.py` would fall back
to a sigmoid of the raw decision score — a rough estimate. Fitting
`CalibratedClassifierCV` (wrapped in `FrozenEstimator`) on the held-out
calibration set fixes this. Lower
Brier score = better-calibrated probabilities, and the reliability table shows
predicted vs. actual spam rates per probability bin.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV, FrozenEstimator
from sklearn.metrics import brier_score_loss
import numpy as np

# sklearn >= 1.6: wrap the prefit pipeline in FrozenEstimator (cv="prefit" was removed in 1.9)
calibrated = CalibratedClassifierCV(FrozenEstimator(best_model))
calibrated.fit(X_cal, y_cal)

raw_proba = 1.0 / (1.0 + np.exp(-best_model.decision_function(X_test)))
cal_proba = calibrated.predict_proba(X_test)[:, 1]
cal_pred = calibrated.predict(X_test)

print(f"Brier score (raw sigmoid):  {brier_score_loss(y_test, raw_proba):.4f}")
print(f"Brier score (calibrated):   {brier_score_loss(y_test, cal_proba):.4f}")

bins = np.linspace(0, 1, 6)
reliability = pd.DataFrame({
    "predicted_mean": [cal_proba[(cal_proba >= lo) & (cal_proba < hi)].mean()
                       for lo, hi in zip(bins[:-1], bins[1:])],
    "actual_rate": [y_test[(cal_proba >= lo) & (cal_proba < hi)].mean()
                    for lo, hi in zip(bins[:-1], bins[1:])],
})
reliability

## 4. Export the best model

Overwrites `models/spam_model.pkl`, which `src/predict.py` and the Flask app
load by default. The calibrated model exposes a trustworthy `predict_proba`,
so the confidence labels in the API reflect real probabilities.

In [ ]:
models_dir = PROJECT_ROOT / "models"
models_dir.mkdir(parents=True, exist_ok=True)
out_path = models_dir / "spam_model.pkl"
joblib.dump(calibrated, out_path)
print(f"Saved calibrated best model to {out_path}")

## 5. Save metrics and confusion matrix to results/

Persist the evaluation for the baseline so `04_transformer_model.ipynb` can
compare against it.

In [ ]:
import json

results_dir = PROJECT_ROOT / "results"
results_dir.mkdir(parents=True, exist_ok=True)

metrics = {
    "model": best_name,
    "calibrated": True,
    "accuracy": round(accuracy_score(y_test, cal_pred), 4),
    "f1_spam": round(f1_score(y_test, cal_pred), 4),
    "roc_auc": round(roc_auc_score(y_test, cal_proba), 4),
    "brier": round(brier_score_loss(y_test, cal_proba), 4),
}
with open(results_dir / "metrics_linear_svm.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(metrics)

(results_dir / "classification_report_linear_svm.txt").write_text(
    classification_report(y_test, cal_pred, target_names=["ham", "spam"])
)

fig, ax = plt.subplots()
ConfusionMatrixDisplay.from_predictions(
    y_test, cal_pred, display_labels=["ham", "spam"], cmap="Blues", ax=ax
)
ax.set_title("Confusion matrix — calibrated Linear SVM")
fig.tight_layout()
fig.savefig(results_dir / "confusion_matrix_linear_svm.png", dpi=150)
plt.show()
print(f"Results saved to {results_dir}")

In [ ]:
from predict import predict  # noqa: E402

for msg in [
    "Congratulations! You've won a FREE iPhone. Click here to claim.",
    "Hey, are we still on for lunch tomorrow at 12:30?",
]:
    result = predict(msg)
    verdict = "SPAM" if result["is_spam"] else "HAM"
    print(f"{verdict:4} | {result['confidence']:6} | {result['spam_probability']:.2f} | {msg}")